# 03 Análisis epxloratorio de reservas

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.carga import cargar_bronze

dfs = cargar_bronze()
dfs.keys()   # agencias, empresas, ocupaciones, reservaciones, segmentos_mercados

In [ ]:
res = dfs["reservaciones"].copy()
emp = dfs["empresas"]

Ver primero lo de las fechas, el formato y cuáles puedan ser o no válidas

In [ ]:
def a_fecha(s, formato="%Y%m%d", patron=r"\d{8}"):
    """Convierte texto a fecha con formato explícito; lo que no encaja queda como NaT.
    El patrón evita que '2019081' se lea como 1 de agosto: %d acepta un solo dígito."""
    texto = s.astype("string").str.strip()
    return pd.to_datetime(texto.where(texto.str.fullmatch(patron)), format=formato, errors="coerce")

In [ ]:
CONVERSION = {"fecha_rsvp": "rsvp", "fecha_registro": "registro", "fecha_llegada": "llegada", "fecha_salida": "salida"}
for original, nueva in CONVERSION.items():
    res[nueva] = a_fecha(res[original])
    invalidas = res[nueva].isna()
    print(f"{original}: {invalidas.sum():,} no convertibles | "
          f"{res[nueva].min().date()} → {res[nueva].max().date()} | "
          f"ejemplos: {res.loc[invalidas, original].unique()[:3].tolist()}")

voy a dejjar las no convertibles quedan como nulo. Las de registro se relacionan con el estatus en el paso 4.9.

# Cuáles son los valores mínimos de fechas de reservas y máximos
esto para futura toma decisión en gráficos de kpis

In [ ]:
pd.DataFrame({c: res[c].agg(["min", "max"]) for c in ["rsvp", "registro", "llegada", "salida", "Fecha_hoy"]}).T

# Qué son ¿`num_noches`, son noches o cuartos noche?

en qué unidad se mide `num_noches`?

Puede ser: `num_noches` es el número de noches de la estancia (salida − llegada).

Voy a comparar con los días entre fechas y con días × habitaciones.

In [ ]:
res = dfs["reservaciones"]

llegada = pd.to_datetime(res["fecha_llegada"], format="%Y%m%d", errors="coerce")
salida = pd.to_datetime(res["fecha_salida"], format="%Y%m%d", errors="coerce")
dias = (salida - llegada).dt.days

igual_dias = (res["num_noches"] == dias).mean()
igual_cuartos = (res["num_noches"] == dias * res["total_habitaciones"]).mean()
print(f"num_noches = días:                {igual_dias:.1%}")
print(f"num_noches = días × habitaciones: {igual_cuartos:.1%}")

In [ ]:
no_cuadra = res[res["num_noches"] != dias * res["total_habitaciones"]]
fecha_invalida = llegada[no_cuadra.index].isna() | salida[no_cuadra.index].isna()
print(f"No cuadran: {len(no_cuadra):,}; con alguna fecha inválida: {fecha_invalida.sum():,}")

In [ ]:
con_hab = res["total_habitaciones"] > 0
print(f"Solo filas con habitaciones: {(res.loc[con_hab, 'num_noches'] == dias[con_hab] * res.loc[con_hab, 'total_habitaciones']).mean():.1%}")

> **Resultado en tu primera exploración:** refutada: `num_noches` son **cuartos noche** (días × habitaciones), la misma unidad que `cuartos_noche` en ocupaciones.
>
> **Decisión:** renombrar a `cuartos_noche` en silver.

# ¿Qué es `estado_huesped`?

In [ ]:
res = dfs["reservaciones"]

con_hab = res["total_habitaciones"] > 0          # quita las fantasma sin usar el desfase
origen = res.loc[con_hab, "estado_huesped"]

print("Valores distintos:", origen.nunique())
origen.value_counts().head(15)

In [ ]:
sorted(origen.unique())[:60]

In [ ]:
print(f"Empiezan por 'E': {origen.str.startswith('E').mean():.1%}")
origen.str.len().value_counts().sort_index()   # longitudes de los códigos

Son una "E" seguida de un código geográfico: AGU (Aguascalientes), WA (Washington), AK (Alaska)... "Estado" significa el estado o provincia de origen del huésped.

# Qué hay de estatus

In [ ]:
res = dfs["reservaciones"].copy()
res["registro"] = pd.to_datetime(res["fecha_registro"], format="%Y%m%d", errors="coerce")
res["llegada"] = pd.to_datetime(res["fecha_llegada"], format="%Y%m%d", errors="coerce")

validas = res[res["total_habitaciones"] > 0]     # quita las fantasma sin usar el desfase
print(f"Reservas: {len(res):,} | con habitaciones: {len(validas):,}")

In [ ]:
validas.groupby("estatus_reservacion_id").agg(
    reservas=("ID_Reserva", "size"),
    sin_registro=("registro", lambda s: s.isna().mean()),
    con_tarifa=("tarifa_total", lambda s: (s > 0).mean()),
    tarifa_suma=("tarifa_total", "sum"),
    cuartos_noche=("num_noches", "sum"))

### Flujo de una reserva hoteleta
confirmada ──> llega (alojado) ──> sale (completada)

Si está confirmada puede:
     ──> cancelada (antes de llegar)
O también: 
     ──> no-show (no se presenta el día de llegada)

Basado en esa hipótesis es que creo el siguiente diccionario:

In [ ]:
GRUPOS = {9: "Completada", 2: "Cancelada / no-show", 5: "Cancelada / no-show",
          1: "Confirmada (futura)", 7: "Alojado", 8: "Alojado", 3: "Otro", 4: "Otro"}
serie = (validas.assign(mes=validas["llegada"].dt.to_period("M"),
                        estado=validas["estatus_reservacion_id"].map(GRUPOS))
                .pivot_table(index="mes", columns="estado", values="num_noches", aggfunc="sum", fill_value=0))
serie.plot(kind="bar", stacked=True, figsize=(14, 5), title="Cuartos noche por mes de llegada y estado")